# 02 — Treinar Xception para real × manipulado

Treina apenas o detector de manipulação. Raça e gênero não são entradas do modelo. Mantemos RGB 299 × 299 e normalização média/desvio 0,5 dos labs enviados. O melhor checkpoint é escolhido pela perda de validação; o teste é reservado para os próximos notebooks.

Use GPU no Colab. Comece pelo piloto apenas para verificar funcionamento; ele é pequeno demais para conclusões de justiça.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/tcc_justica')
ROOT.mkdir(parents=True, exist_ok=True)
print('Pasta persistente:', ROOT)


In [ ]:
%pip -q install timm==1.0.22 scikit-learn pandas matplotlib scipy pillow


In [ ]:
import os, random, json, hashlib, io
import numpy as np, pandas as pd
import torch, timm
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ARCH='legacy_xception'
SIZE=299
class Faces(Dataset):
    def __init__(self,df,augment=False):
        self.df=df.reset_index(drop=True)
        self.tf=transforms.Compose([transforms.Resize((SIZE,SIZE))]+([transforms.RandomHorizontalFlip()] if augment else [])+[transforms.ToTensor()])
    def __len__(self):return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        with Image.open(r.image_path) as im:x=self.tf(im.convert('RGB'))
        return x,int(r.label),i

def normalized(x):return (x-.5)/.5

def load_trained(path):
    assert Path(path).is_file(), 'Falta checkpoint treinado do notebook 02.'
    ck=torch.load(path,map_location='cpu',weights_only=True)
    assert ck.get('class_map')=={'real':0,'fake':1}, 'Checkpoint incompatível; use notebook 02.'
    assert ck['mean']==[.5]*3 and ck['std']==[.5]*3 and ck['size']==299
    model=timm.create_model(ck['arch'],pretrained=False,num_classes=2)
    model.load_state_dict(ck['state_dict'],strict=True)
    return model.to(DEVICE).eval(),ck

def check_manifest(df):
    assert df.image_id.is_unique
    assert set(df.label)=={0,1} and set(df.split)=={'train','val','test'}
    assert df.groupby('unit_id').split.nunique().max()==1
    assert df.groupby('pair_id').split.nunique().max()==1
    owners={}
    for r in df.itertuples():
        for u in [r.unit_id,r.source_unit_id]:
            assert owners.setdefault(u,r.split)==r.split, 'Fonte/alvo cruzam splits.'
    assert df.image_path.map(lambda p:Path(p).is_file()).all(), 'Há imagens ausentes.'

def manifest_hash(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()


## 1. Configuração e integridade


In [ ]:
DATASET=ROOT/'geracoes'/'piloto_10'/'dataset.csv' # ajuste após gerar a base completa
RUN='xception_piloto_10'
MODEL_DIR=ROOT/'modelos'/RUN;MODEL_DIR.mkdir(parents=True,exist_ok=True)
CHECKPOINT=MODEL_DIR/'detector.pt'
assert not CHECKPOINT.exists(), 'Use outro RUN para preservar o experimento anterior.'
df=pd.read_csv(DATASET);check_manifest(df)
loaders={s:DataLoader(Faces(df[df.split==s],augment=s=='train'),batch_size=16,
                     shuffle=s=='train',num_workers=2) for s in ['train','val']}
display(df.groupby(['split','label']).size())
print('Dispositivo:',DEVICE)


## 2. Treino e seleção pela validação
O transfer learning começa com pesos ImageNet, mas o modelo só é avaliado como detector depois do treino supervisionado real/fake. Nunca usar uma nova cabeça aleatória como evidência de viés. O teste não é usado para escolher épocas, limiar ou hiperparâmetros.


In [ ]:
model=timm.create_model(ARCH,pretrained=True,num_classes=2).to(DEVICE)
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-5,weight_decay=1e-4)
criterion=nn.CrossEntropyLoss()
EPOCHS=10;PATIENCE=3;best=float('inf');waiting=0;history=[]
for epoch in range(1,EPOCHS+1):
    row={'epoch':epoch}
    for phase in ['train','val']:
        model.train(phase=='train');loss_sum=correct=n=0
        for x,y,_ in loaders[phase]:
            x,y=x.to(DEVICE),y.to(DEVICE)
            with torch.set_grad_enabled(phase=='train'):
                logits=model(normalized(x));loss=criterion(logits,y)
                if phase=='train':
                    optimizer.zero_grad(set_to_none=True);loss.backward();optimizer.step()
            loss_sum+=loss.item()*len(y);correct+=(logits.argmax(1)==y).sum().item();n+=len(y)
        row[phase+'_loss']=loss_sum/n;row[phase+'_acc']=correct/n
    history.append(row);print(row)
    if row['val_loss']<best:
        best=row['val_loss'];waiting=0
        torch.save({'state_dict':model.state_dict(),'arch':ARCH,'size':SIZE,
            'mean':[.5]*3,'std':[.5]*3,'class_map':{'real':0,'fake':1},
            'manifest_sha256':manifest_hash(DATASET),'seed':SEED,'epoch':epoch,
            'timm_version':str(timm.__version__),'torch_version':str(torch.__version__)},CHECKPOINT)
    else:waiting+=1
    pd.DataFrame(history).to_csv(MODEL_DIR/'historico.csv',index=False)
    if waiting>=PATIENCE:break
print('Melhor checkpoint:',CHECKPOINT)


In [ ]:
import matplotlib.pyplot as plt
h=pd.DataFrame(history)
fig,axs=plt.subplots(1,2,figsize=(11,4))
for phase in ['train','val']:
    axs[0].plot(h.epoch,h[phase+'_loss'],label=phase)
    axs[1].plot(h.epoch,h[phase+'_acc'],label=phase)
axs[0].set_title('Perda');axs[1].set_title('Acurácia')
for ax in axs:ax.set_xlabel('Época');ax.legend()
plt.tight_layout();plt.savefig(MODEL_DIR/'treino.png',dpi=200);plt.show()
